# Testing accuracy of cellprofiler feature extraction for cell size - seeing which method is best

In [ ]:
import os
import numpy as np
import pandas as pd
import sqlite3
#plotting
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from scipy.stats import shapiro
import re
from scipy import stats
from pathlib import Path
from helpers import *
from plate_preprocessing import *
from mitolyso_plot_functions import *

In [ ]:
## Import your csv files
csvpath = '/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs/'
filename = 'total_combined_cell.csv'
combined_cell_df_mitolyso = pd.read_csv(os.path.join(csvpath, filename))
#filter_df = cell_filters(combined_cell_df_mitolyso)
#display(combined_cell_df_mitolyso.shape)

stitched_path = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/stitching/segmentation_testset"
stitched_csv = "stitched_test_data_v2.csv"
stitched_csv_borders_excluded = "stitched_test_data_v2_borders_excluded.csv"

stitched_cells_df = pd.read_csv(os.path.join(stitched_path,stitched_csv))
stitched_cells_df_borders_excluded = pd.read_csv(os.path.join(stitched_path, stitched_csv_borders_excluded))

feature_meas = "Cell_AreaShape_Area"

In [ ]:
combined_cell_df_mitolyso = combined_cell_df_mitolyso[combined_cell_df_mitolyso['Staining'].str.startswith("LAMP1-488 + MitoRed")]
min_x = 0
min_y = 0
max_x = combined_cell_df_mitolyso["Image_Width_DAPI"][0] # get the max x and y resolutions
max_y = combined_cell_df_mitolyso["Image_Height_DAPI"][0]

outpath = stitched_path

combined_cell_df_mitolyso_borders_excluded = exclude_borders(combined_cell_df_mitolyso, min_x, min_y, max_x, max_y, prefix="Cell_")
combined_cell_df_mitolyso_borders_excluded.to_csv(os.path.join(outpath, "total_combined_cell_borders_excluded.csv"), index=False)
combined_cell_df_mitolyso_borders_excluded.describe().to_csv(os.path.join(outpath, "total_combined_cell_borders_excluded_stats.csv"))
combined_cell_df_mitolyso_borders_excluded.groupby("AllGroups")["Cell_AreaShape_Area"].describe().to_csv(os.path.join(outpath, "total_combined_cell_borders_excluded_passage_group_stats.csv"))

## Prep the stitched cell dataframe

In [ ]:
#rescale area to the original image by multiplying by the inverse of the rescale factor squared, e.g. for a factor of 1/4 then I can multipy by (4^2)=16
#original_area = measured_area * (1 / rescale_factor**2)

def find_replicate(path):
    replicate_pattern = r"R(\d{1})"  # Matches "RX" where X is the replicate number (placeholder for now)
    match = re.search(replicate_pattern, path)
    if match:
        replicate = int(match.group(1))
    else:
        replicate = None 
    return replicate

def find_row_col(well_code):
    rowcol_pattern = r"r(\d{1,2})c(\d{1,2})"  # Matches "RX" where X is the replicate number (placeholder for now)
    match = re.search(rowcol_pattern, well_code)
    if match:
        row_metadata = int(match.group(1))
        col_metadata = int(match.group(2))
    else:
        row_metadata = None 
        col_metadata = None
    return row_metadata,col_metadata

def prepare_stitched_cells_df_v2(stitched_cells_df, feature_meas = "Cell_AreaShape_Area"):
    stitched_cells_df["Replicate_Number"] = stitched_cells_df.apply(
            lambda row: find_replicate(
                path=row["Path"]
            ), axis=1)

    unique_replicates = stitched_cells_df["Replicate_Number"].unique()
    metadata_sliced_dfs = [] #basicalyl split into 3 and rejoin them
    for i,rep in enumerate(unique_replicates):
        if rep == 5:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250328_rep05_metadata/map.csv"
        elif rep == 6:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250410_rep06_metadata/map.csv"
        elif rep == 7:
            map_file="/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250501_rep07_metadata/map.csv"
        if os.path.exists(map_file):
            platemap_df = pd.read_csv(map_file)
            platemap_df = platemap_df.drop_duplicates(subset=['Metadata_WellRow', 'Metadata_WellColumn']) #drop the dupes here, its ok bcus we only need the passage num
            platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
            platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)

            #join the metadata and add to a list to join
            stitched_cells_df_prefilter = stitched_cells_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn'], how="left")
            stitched_cells_df_filter = stitched_cells_df_prefilter[stitched_cells_df_prefilter["Replicate_Number"]==rep]
            #display(stitched_cells_df_filter)
            metadata_sliced_dfs.append(stitched_cells_df_filter)
    #display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    stitched_cells_df = pd.concat(metadata_sliced_dfs)
    stitched_cells_df["Passage Group"] = stitched_cells_df['PassageNumber'].apply(passage_group)
    stitched_cells_df["AllGroups"] = add_drug_to_group(stitched_cells_df, "Passage Group", "Drug")
    return stitched_cells_df
    #display(stitched_cells_df)

def prepare_stitched_cells_df_v1(stitched_cells_df, feature_meas = "Cell_AreaShape_Area"):
    stitched_cells_df["Replicate_Number"] = stitched_cells_df.apply(
            lambda row: find_replicate(
                path=row["Path"]
            ), axis=1)
    stitched_cells_df[["Metadata_WellRow","Metadata_WellColumn"]] = stitched_cells_df.apply(
            lambda row: find_row_col(
                well_code=row["Well_id"]
            ), axis=1, result_type="expand")
    #rescale the area by a factor of 16 (inverse of 0.25^2)
    stitched_cells_df["Cell_AreaShape_Area"] = stitched_cells_df.apply(lambda x: x["area"]*16, axis=1)
    display(stitched_cells_df)

    unique_replicates = stitched_cells_df["Replicate_Number"].unique()
    metadata_sliced_dfs = [] #basicalyl split into 3 and rejoin them
    for i,rep in enumerate(unique_replicates):
        if rep == 5:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250328_rep05_metadata/map.csv"
        elif rep == 6:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250410_rep06_metadata/map.csv"
        elif rep == 7:
            map_file="/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250501_rep07_metadata/map.csv"
        if os.path.exists(map_file):
            platemap_df = pd.read_csv(map_file)
            platemap_df = platemap_df.drop_duplicates(subset=['Metadata_WellRow', 'Metadata_WellColumn']) #drop the dupes here, its ok bcus we only need the passage num
            platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
            platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)

            #join the metadata and add to a list to join
            stitched_cells_df_prefilter = stitched_cells_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn'], how="left")
            stitched_cells_df_filter = stitched_cells_df_prefilter[stitched_cells_df_prefilter["Replicate_Number"]==rep]
            #display(stitched_cells_df_filter)
            metadata_sliced_dfs.append(stitched_cells_df_filter)
    #display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    stitched_cells_df = pd.concat(metadata_sliced_dfs)
    stitched_cells_df["Passage Group"] = stitched_cells_df['PassageNumber'].apply(passage_group)
    stitched_cells_df["AllGroups"] = add_drug_to_group(stitched_cells_df, "Passage Group", "Drug")
    return stitched_cells_df
    #display(stitched_cells_df)

In [ ]:
#display(stitched_cells_df)
stitched_cells_df = prepare_stitched_cells_df_v2(stitched_cells_df)
stitched_csv_borders_excluded = prepare_stitched_cells_df_v2(stitched_cells_df_borders_excluded)
display(stitched_cells_df)

group_avg_df = average_groups_by_plate(combined_cell_df_mitolyso, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
group_avg_df_borders_excluded = average_groups_by_plate(combined_cell_df_mitolyso_borders_excluded, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
group_avg_df = apply_shapiro_wilk_test_to_df(group_avg_df,feature_meas)
group_avg_df_borders_excluded = apply_shapiro_wilk_test_to_df(group_avg_df_borders_excluded,feature_meas)
display(group_avg_df_borders_excluded)



## code to make the side-by-side comparison plots

In [ ]:
def average_groups_pivot(group_avg_df, x_value, y_value, replicate_col_name):
    """Make a pivot table from the averaged dataframe

    Args:
        group_avg_df (DataFrame): your dataframe output from average_groups_by_plate()
        x_value (string): the grouping variable (x value)
        y_value (string): the quantitavie feature to measure (y value)
        replicate_col_name (string): the variable representing experimental replicates for grouping

    Returns:
        DataFrame: a pivot table
    """    
    group_avg_df_pivot = group_avg_df.pivot_table(columns=x_value, values=y_value, index=replicate_col_name)
    return group_avg_df_pivot

def tukey_statsmodels(data, test_groups, feature):
    """
    Perform a oneway anova test and a pairwise tukey post hoc test using averaged values per replicate
    Returns a dataframe
    """
    from statsmodels.stats.multicomp import pairwise_tukeyhsd

    df = data.copy()
    # groups = getpairs(temp_copy, 'Passage Group')
    # calculate tukey HSD
    tukey = pairwise_tukeyhsd(endog=df[feature], groups=df[test_groups], alpha=0.05)

    # Extract relevant results
    tukey_results = np.array(tukey.summary().data)
    return tukey_results

def pvalues_anova_and_tukeyhsd_posthoc(data_df, pivot_df, x_value, y_value, replicate_number_col="Replicate_Number", desired_pairs=None, order=None):
    """Perform Tukey's HSD post-hoc test on the data.
    See https://github.com/4dcu-be/CodeNuggets/blob/main/Post%20hoc%20tests%20with%20statannotations.ipynb 
    Also https://www.biorxiv.org/content/10.1101/2025.02.02.636071v1.full.pdf 
    Args:
        data_df (pd.DataFrame): DataFrame table containing the data.
        pivot_df (pd.DataFrame): DataFrame pivot table containing the means.
        x_value (str): Column name for the independent variable.
        y_value (str): Column name for the dependent variable.
        grouping_variable (str): Column name for the replicate number.
        order (list, optional): Order of groups for plotting. Defaults to None.

    Returns:
        pd.DataFrame: DataFrame with Tukey's HSD results.
    """
    from scipy.stats import f_oneway
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    groups = []  # Convert pivot table to list of groups
    display(pivot_df)
    for col in pivot_df: 
        if col == x_value or col == replicate_number_col: 
            print("Skipping col:" +col)
            continue# Skip the first column (usually the index or grouping variable)
        else:
            print("Adding col:" +col)
            groups.append(pivot_df[col].dropna())
    # One-Way ANOVA
    display(groups)
    f_value, p_value_anova = f_oneway(*list(groups)) #Pass groups as args to run ANOVA on all groups
    print(f'ANOVA F statistic: {f_value}')
    print(f'ANOVA p value: {p_value_anova}')

    # Tukey's HSD (post hoc test)
    if p_value_anova < 0.05:
        tukey_result = pairwise_tukeyhsd(endog=data_df[y_value], groups=data_df[x_value], alpha=0.05)
        # Extract the data from the Statsmodels SimpleTable
        tukey_data = tukey_result._results_table.data[1:]  # Exclude the header line
        headers = tukey_result._results_table.data[0]  # Get the header line
        tukey_result_df = pd.DataFrame(tukey_data, columns=headers)
        tukey_result_pairs = tukey_result_df[['group1', 'group2']].itertuples(index=False, name=None)
        pairs = list(tukey_result_pairs)
        p_values = tukey_result_df['p-adj'].tolist()
        display(tukey_result_df)
        return (pairs, p_values)
    else:
        print("ANOVA test is not significant, skipping Tukey's HSD post-hoc test.")
        return ([], [])
    

def shapiro_pvalue(group_avg_df, replicate, feature_meas, replicate_col_name ="Replicate_Number", debug=False):
    """Function to apply the shapiro wilk test to a dataframe aggregated by replicate for a single feature

    Args:
        group_avg_df (DataFrame): the aggregated dataframe
        replicate (string, int): string or int representation of replicate number
        feature_meas (string): _description_
        replicate_col_name (str, optional): the name of the replicate column. Defaults to "Replicate_Number".
        debug (bool, optional): print out p values. Defaults to False.

    Returns:
        float: p_value from test on that replicate
    """    
    from scipy.stats import shapiro
    rep_df = group_avg_df[group_avg_df[replicate_col_name] == replicate]
    # Assume 'df' is your DataFrame and 'feature_meas' is the column to test
    stat, p_value = shapiro(rep_df[feature_meas].dropna())
    
    if debug:
        print(f"Shapiro-Wilk statistic: {stat}, p-value: {p_value}")
        if p_value < 0.05:
            print("Data is not normally distributed (reject H0)")
        else:
            print("Data is normally distributed (fail to reject H0)")
    return p_value

def apply_shapiro_wilk_test_to_df(group_avg_df,feature_meas, replicate_col_name ="Replicate_Number", alpha = 0.05):
    """Function to applies the shapiro wilk test row-by-row onto an aggregated dataframe by replicate

    Args:
        group_avg_df (DataFrame): the aggregated dataframe
        feature_meas (string): _description_
        replicate_col_name (str, optional): the name of the replicate column. Defaults to "Replicate_Number".
        alpha (float): the p value threshold. Defaults to p=0.05

    Returns:
        DataFrame: The aggregated dataframe with a "Shaprio_pvalue" column and a boolean "Shapiro_normality" column
    """    
    #apply the shapiro-wilk test to a dataframe
    group_avg_df = group_avg_df.dropna()
    group_avg_df["Shapiro_pvalue"] = group_avg_df.apply(
        lambda row: shapiro_pvalue(
            group_avg_df,
            replicate=row[replicate_col_name],
            feature_meas=feature_meas
        ), axis=1)
    #reject null hypothesis if p < 0.05 - i.e. significant chance that the data is not normally distributed
    group_avg_df["Shapiro_normality"] = group_avg_df.apply(
        lambda row: 
            row['Shapiro_pvalue'] > alpha,
        axis=1)
    return group_avg_df


In [ ]:
def visualize_descriptive_stats(df, groups, feature, replicate_number_col, alpha=0.05):
    unique_groups = df[groups].unique()
    num_groups = len(df[groups].unique())
    num_selected_cols = len(df.columns)
    num_rows = (num_selected_cols + num_groups - 1) // num_groups

    fig, axs = plt.subplots(nrows=num_rows, ncols=num_groups, figsize=(15, 5 * num_rows))
    result = []

    for i, group in enumerate(unique_groups):
        if group not in df.columns:
            continue

        treatment_group = df[df[feature,replicate_number_col][df[groups] == group]]
        print(treatment_group, group)
        descriptive_stats = treatment_group.describe()
        shapiro_stat, shapiro_p = shapiro(treatment_group)

        interpretation = 'Distribution looks Gaussian (fail to reject H0)' if shapiro_p > alpha else 'Distribution does not look Gaussian (reject H0)'
        interp = 'Looks Gaussian \n(fail to reject H0)' if shapiro_p > alpha else 'Does not look Gaussian \n(reject H0)'

        row = {
            'Group': group,
            'Count': descriptive_stats['count'],
            'Mean': descriptive_stats['mean'],
            'Std': descriptive_stats['std'],
            'Min': descriptive_stats['min'],
            '25%': descriptive_stats['25%'],
            '50%': descriptive_stats['50%'],
            '75%': descriptive_stats['75%'],
            'Max': descriptive_stats['max'],
            'Alpha': alpha,
            'Shapiro W': shapiro_stat,
            'Shapiro p': shapiro_p,
            'Interpretation': interpretation
        }
        result.append(row)

        row_idx = i // num_groups
        col_idx = i % num_groups
        ax = axs[row_idx, col_idx]
        
        # # Set histogram color if specified
        # color_index = i % len(hist_colors) if hist_colors else 0
        color = 'purple' #hist_colors[color_index] if hist_colors else 'purple'
        
        sns.histplot(data=treatment_group, x=feature, hue = replicate_number_col, kde=True, alpha=0.8, ax=ax)
        ax.set_title(group, fontsize=12)
        ax.set_xlabel('')
        ax.text(0.97, 0.92, f'Statistic={shapiro_stat:.4f}\np={shapiro_p:.4f}\n{interp}',
                fontsize=10, ha='right', va='top', transform=ax.transAxes)

    # Remove empty subplots
    if num_selected_cols < num_rows * num_groups:
        for i in range(num_selected_cols, num_rows * num_groups):
            axs.flat[i].set_visible(False)

    plt.tight_layout()
    plt.show()
    return pd.DataFrame(result)
#visualize_descriptive_stats(stitched_cells_df, groups="AllGroups", feature="Cell_AreaShape_Area", replicate_number_col="Replicate_Number", alpha=0.05)

In [ ]:
def annotate_with_anova_tukey(ax, pairs, data, x_value, y_value, replicate_col_name = "Replicate_Name", order=None, plot="violinplot"):
    """Add statistical annotations to the plot using one-way ANOVA test.
    see https://statannotations.readthedocs.io/en/latest/custom-test.html for more examples

    Args:
        ax (_type_): _description_
        pairs (_type_): _description_
        group_avg_df (_type_): _description_
        x_value (_type_): _description_
        y_value (_type_): _description_
        order (_type_, optional): _description_. Defaults to None.
        plot (str, optional): _description_. Defaults to "violinplot".

    Returns:
        _type_: _description_
    """    
    from statannotations.Annotator import Annotator, StatTest
    from scipy.stats import tukey_hsd

    custom_long_name = 'Pairwise Tukey HSD'
    custom_short_name = 'tukey'
    custom_func = tukey_hsd
    tukey = StatTest(custom_func, custom_long_name, custom_short_name)
    #tukey = StatTest(tukey_hsd, custom_long_name, custom_short_name)
    
    #load the custom test
    annotator = Annotator(ax, pairs, data=data, order=order, plot=plot)#x=x_value, y=y_value, hue=replicate_col_name, 
    annotator.reset_configuration() 
    annotator.configure(test=tukey, 
                        text_format='star', #'simple','full'
                        loc='inside', 
                        hide_non_significant = True,
                        color = 'black',
                        verbose = 2)
    annotator.apply_and_annotate()
    return ax

def annotate_with_tukey_pvalues(ax, data, pivot_data, x_value, y_value, replicate_col_name="Replicate_Name", pairs=None, order=None, plot="violinplot"):
    """Add statistical annotations to the plot using Tukey's HSD test.
    see https://statannotations.readthedocs.io/en/latest/custom-test.html for more examples

    Args:
        ax (_type_): _description_
        pairs (_type_): _description_
        data (_type_): _description_
        x_value (_type_): _description_
        y_value (_type_): _description_
        order (_type_, optional): _description_. Defaults to None.
        plot (str, optional): _description_. Defaults to "violinplot".

    Returns:
        _type_: _description_
    """    
    from statannotations.Annotator import Annotator
    used_pairs, p_values = pvalues_anova_and_tukeyhsd_posthoc(data, pivot_data, x_value, y_value, order=order, desired_pairs=pairs)
    if used_pairs is None or len(used_pairs) == 0:
        print("No significant pairs found for Tukey's HSD test.")
        return ax
    else:
        annotator = Annotator(ax=ax, pairs=list(used_pairs), data=data, plot=plot,x=x_value, y=y_value, order=order)
        annotator.reset_configuration()
        annotator.configure(text_format='simple', 
                            test_short_name='tukey',
                            #pvalue_format = [[1e-5, "1e-5"], [1e-4, "1e-4"], [1e-3, "0.001"], [1e-2, "0.01"], [5e-2, "0.05"]],
                            loc='inside', 
                            hide_non_significant = True,
                            color = 'black',
                            verbose = 2)
        annotator.set_pvalues_and_annotate(p_values)
        return ax

def annotate_with_kruskal(ax, pairs, data, x_value, y_value, replicate_col_name="Replicate_Name", order=None, plot="violinplot"):
    """Add statistical annotations to the plot using Kruskal-Wallis test.
    see https://statannotations.readthedocs.io/en/latest/custom-test.html for more examples

    Args:
        ax (_type_): _description_
        pairs (_type_): _description_
        data (_type_): _description_
        x_value (_type_): _description_
        y_value (_type_): _description_
        order (_type_, optional): _description_. Defaults to None.
        plot (str, optional): _description_. Defaults to "violinplot".

    Returns:
        _type_: _description_
    """    
    from statannotations.Annotator import Annotator
    annotator = Annotator(ax, pairs=pairs, data=data, order=order, plot=plot, x=x_value, y=y_value) 
    annotator.reset_configuration() 
    annotator.configure(test='Kruskal', 
                        text_format='simple', 
                        #pvalue_format = [[1e-5, "1e-5"], [1e-4, "1e-4"], [1e-3, "0.001"], [1e-2, "0.01"], [5e-2, "0.05"]],
                        loc='inside', 
                        hide_non_significant = True,
                        color = 'black',
                        verbose = 2)
    annotator.apply_and_annotate()
    return ax
    
def annotate_legend_with_shapiro(ax, group_avg_df, group_col_name, shapiro_col_name="Shapiro_normality", palette='pastel', title ="Replicate"):
    """add an annotation to the legend of an axis if there is normality via shapiro test

    Args:
        ax (_type_): _description_
        group_avg_df (_type_): _description_
        replicate_col_name (_type_): _description_
    """  
    import matplotlib.lines as mlines  
    unique_replicates = group_avg_df[group_col_name].unique()
    L = plt.legend()
    custom_labels = []
    for rep in unique_replicates:
        label = str(rep)
        shapiro_val = group_avg_df[group_avg_df[group_col_name] == rep][shapiro_col_name].iloc[0]
        if shapiro_val:
            label += " (normal)"
        custom_labels.append(label)

    # Create custom legend handles (using the same colors as swarmplot)
    palette = sns.color_palette(palette, n_colors=len(unique_replicates))
    handles = [
        mlines.Line2D([], [], color=palette[i], 
                      marker='o', 
                      linestyle='None', 
                      markersize=12, 
                      markeredgecolor='black', 
                      label=custom_labels[i])
        for i in range(len(unique_replicates))]
    ax.legend_.set_title(title)
    ax.legend(handles=handles, title=title, loc="best")
    return ax

def superviolinplot_helper(data_df, group_avg_df, ax, x_value, y_value, title, replicate_col_name, pairs=None, order=None, annotate=False, test=None):
    if pairs is None:
        pairs = getpairs(data_df, x_value, order=order)
    print(pairs)
    sns.violinplot(
        data=data_df, 
        x=x_value, y=y_value, #hue=x_value,
        #palette="Set2", 
        split=True, #using split violin plots - only one side, basically looks like a histogram
        inner ="quart",
        color="gainsboro",
        width=0.9, 
        linewidth=1.5,
        order=order,
        ax=ax
    )
    sns.swarmplot(
        data=group_avg_df, x=x_value, y=y_value,
        hue=replicate_col_name, 
        order=order, 
        palette='pastel',
        size=12, 
        edgecolor="k", 
        linewidth=1, 
        dodge=False, 
        ax=ax
    )
    #draw a boxplot to show the mean line
    sns.boxplot(data=group_avg_df, x=x_value, y=y_value,
        showmeans=True,
        meanline=True,
        meanprops={'color': 'dimgray', 'ls': '-', 'lw': 2.5},
        medianprops={'visible': False},
        whiskerprops={'visible': False},
        zorder=2,
        showfliers=False,
        showbox=False,
        showcaps=False,
        ax=ax)
    ax.set_title(title)
    
        # axes[0].text(
        #     x=row[x_value], 
        #     y=row[y_value], 
        #     s=str(row["Shapiro_normality"]), 
        #     color="black", 
        #     fontsize=10,
        #     ha="center"
        # )
        #use pivot table to get the average values for each group
    if annotate and test is not None:
        group_avg_pivot_table = average_groups_pivot(group_avg_df, x_value, y_value, replicate_col_name)
        if test == "tukey" or test == "anova":
            ax = annotate_with_tukey_pvalues(ax, group_avg_df, group_avg_pivot_table, x_value, y_value, replicate_col_name=replicate_col_name, order=order, plot="violinplot")
            #ax = annotate_with_anova_tukey(ax, pairs, group_avg_df_pivot, x_value, y_value, replicate_col_name=replicate_col_name, order=order, plot="violinplot")
        elif test == "kruskal":
            ax = annotate_with_kruskal(ax, pairs, group_avg_pivot_table, x_value, y_value, order=order, replicate_col_name=replicate_col_name, plot="violinplot")
        ax = annotate_legend_with_shapiro(ax, group_avg_df, replicate_col_name)
        
    return ax

def superplot_for_area_threshold_comparisons(data_df_1, group_avg_df_1, data_df_2, group_avg_df_2, x_value="AllGroups", y_value="Cell_AreaShape_Area", replicate_col_name = "Replicate_Number", out_dir="", xtitle=None, ytitle=None, order = None, legend=True, title1 = "Original Dataset", title2="Excluding Cells Touching Borders", annotate = False, test=None, export_pivot=False, show_hist=False):
    """Make two side-by-side superplots to compare area between different conditions
    Args:
        data_df_1 (_type_): _description_
        group_avg_df_1 (_type_): _description_
        data_df_2 (_type_): _description_
        group_avg_df_2 (_type_): _description_
        x_value (str, optional): _description_. Defaults to "AllGroups".
        y_value (str, optional): _description_. Defaults to "Cell_AreaShape_Area".
        replicate_col_name (str, optional): _description_. Defaults to "Replicate_Number".
        csv_dir (str, optional): _description_. Defaults to "".
        xtitle (_type_, optional): _description_. Defaults to None.
        ytitle (_type_, optional): _description_. Defaults to None.
    """
    import matplotlib.lines as mlines
    from statannotations.Annotator import Annotator
    from statannotations.stats.StatTest import StatTest
    if order == None:
        order = get_all_group_order()
    pairs = getpairs(data_df_1,x_value,order=order)
    print(pairs)
        
    if show_hist:
        hist = sns.kdeplot(data_df_1, x=y_value, hue =replicate_col_name, palette='pastel')
        plt.show()
        plt.close(hist.figure)
    fig, axes = plt.subplots(1, 2, figsize=(30, 10), sharey=True, sharex=False)
    #plt.style.use("ggplot")
    sns.set_context("talk", font_scale=1.2)
    sns.set_theme(style="whitegrid")
    
    #First subplot: Full Dataset
    # sns.stripplot(
    #     data=data_df_1, 
    #     x=x_value, y=y_value, hue=x_value,
    #     palette="Set2", 
    #     order=order,
    #     ax=axes[0],
    # )
    axes[0] = superviolinplot_helper(data_df_1, group_avg_df_1, axes[0], 
                                     x_value, y_value, 
                                     title1, 
                                     replicate_col_name, 
                                     order=order,
                                     annotate=annotate,
                                     test=test)
    axes[1] = superviolinplot_helper(data_df_2, group_avg_df_2, axes[1], 
                                     x_value, y_value, 
                                     title2, 
                                     replicate_col_name, 
                                     order=order, 
                                     annotate=annotate,
                                     test=test)
    
    axes[0].set_title(title1)
    axes[1].set_title(title2)
    
    if legend:
        axes[0] = annotate_legend_with_shapiro(axes[0], group_avg_df_1, replicate_col_name)
        axes[1] = annotate_legend_with_shapiro(axes[1], group_avg_df_2, replicate_col_name)
    else:
        axes[0].legend_.remove() 
    if ytitle is not None:
        axes[0].set_ylabel(ytitle)  
    if xtitle is not None:
        axes[0].set_xlabel(xtitle)  
        axes[1].set_xlabel(xtitle) 
    
    
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"combined_cellsize_boxplots_{title2}.png"))
    plt.show()
    if export_pivot:
        group_avg_df_1_pivot = average_groups_pivot(group_avg_df_1, x_value, y_value, replicate_col_name)
        group_avg_df_2_pivot = average_groups_pivot(group_avg_df_2, x_value, y_value, replicate_col_name)
        group_avg_df_1_pivot.to_csv(os.path.join(out_dir,f"area_pivot_{title1}.csv")) #can plop this into graphpad and see what it tells me
        group_avg_df_2_pivot.to_csv(os.path.join(out_dir,f"area_pivot_{title2}.csv"))

    
superplot_for_area_threshold_comparisons(combined_cell_df_mitolyso, group_avg_df, 
                                         combined_cell_df_mitolyso_borders_excluded, group_avg_df_borders_excluded, 
                                         x_value="AllGroups", 
                                         y_value="Cell_AreaShape_Area", 
                                         out_dir=stitched_path, 
                                         xtitle = "Passage Groups", 
                                         ytitle="Cell Area", 
                                         title1="From Original Dataset", 
                                         title2="Borders Excluded", 
                                         annotate = True, 
                                         test="tukey",
                                         export_pivot=True,
                                         show_hist=True
                                         )


## Make plots and csvs for the comparison btwn regular images and the stitched images

In [ ]:
stitch_group_avg_df = average_groups_by_plate(stitched_cells_df, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
stitch_group_avg_df = apply_shapiro_wilk_test_to_df(stitch_group_avg_df,"Cell_AreaShape_Area")
outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/stitching/segmentation_testset/data_summarystats"
#stitch_group_avg_df_borders_excluded = average_groups_by_plate(stitched_cells_df_borders_excluded, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
#stitch_group_avg_df_borders_excluded = apply_shapiro_wilk_test_to_df(stitch_group_avg_df_borders_excluded,"Cell_AreaShape_Area")

combined_cell_df_mitolyso["Metadata_Well"] = combined_cell_df_mitolyso.apply(lambda x: well_namer(x["Metadata_WellRow"],x["Metadata_WellColumn"]), axis=1)
combined_cell_df_mitolyso_subset_extrawells = combined_cell_df_mitolyso[combined_cell_df_mitolyso["TimepointName"].isin(stitched_cells_df["TimepointName"])]
combined_cell_df_mitolyso_subset = combined_cell_df_mitolyso_subset_extrawells[combined_cell_df_mitolyso_subset_extrawells["Metadata_Well"].isin(stitched_cells_df["Metadata_Well_x"])]

group_avg_df_subset = average_groups_by_plate(combined_cell_df_mitolyso_subset, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
group_avg_df_subset = apply_shapiro_wilk_test_to_df(group_avg_df_subset,"Cell_AreaShape_Area")

display(group_avg_df_subset)
display(stitch_group_avg_df)

superplot_for_area_threshold_comparisons(combined_cell_df_mitolyso_subset, group_avg_df_subset, 
                                         stitched_cells_df, stitch_group_avg_df, 
                                         out_dir=outpath, 
                                         x_value="AllGroups",
                                         y_value="Cell_AreaShape_Area",
                                         order=["P6-10","P17-19","P29+","Doxo"], 
                                         title2="From Stitched Images", 
                                         annotate = True, 
                                         test="tukey",
                                         export_pivot=True,
                                         xtitle="Passage Groups",
                                         ytitle="Cell Area")

stitched_cells_df.to_csv(os.path.join(outpath, "stitched_cells.csv"), index=False)
stitched_cells_df.describe().to_csv(os.path.join(outpath, "stitched_cell_stats.csv"))
stitched_cells_df.groupby("AllGroups")["Cell_AreaShape_Area"].describe().to_csv(os.path.join(outpath, "stitched_cell_passage_group_stats.csv"))


In [ ]:
print(pvalues_anova_and_tukeyhsd_posthoc(
    data_df=stitched_cells_df,
    pivot_df=average_groups_pivot(stitch_group_avg_df, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicate_col_name="Replicate_Number"),
    x_value='AllGroups',
    y_value="Cell_AreaShape_Area",
    replicate_number_col="Replicate_Number",
    order=None,
))

In [ ]:
stitched_cells_df_borders_excluded = prepare_stitched_cells_df_v2(stitched_cells_df_borders_excluded)
stitch_group_avg_df_borders_excluded = average_groups_by_plate(stitched_cells_df_borders_excluded, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')

stitch_group_avg_df_borders_excluded = average_groups_by_plate(stitched_cells_df_borders_excluded, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
stitch_group_avg_df_borders_excluded = apply_shapiro_wilk_test_to_df(stitch_group_avg_df_borders_excluded,"Cell_AreaShape_Area")

superplot_for_area_threshold_comparisons(combined_cell_df_mitolyso_subset, group_avg_df_subset, stitched_cells_df, stitch_group_avg_df, order=["P6-10","P17-19","P29+","Doxo"], title1="From Stitched Images", title2="Borders Excluded", annotate = True)

outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/stitching/segmentation_testset/data_summarystats"

stitched_cells_df_borders_excluded.to_csv(os.path.join(outpath, "stitched_cells_borders_excluded.csv"), index=False)
stitched_cells_df_borders_excluded.describe().to_csv(os.path.join(outpath, "stitched_cells_borders_excluded_stats.csv"))
stitched_cells_df_borders_excluded.groupby("AllGroups")["Cell_AreaShape_Area"].describe().to_csv(os.path.join(outpath, "stitched_cell_borders_excluded_passage_group_stats.csv"))
